In [ ]:
!pip install opencv-python

In [ ]:
import os
import cv2
import numpy as np
from tqdm import tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
def ben_graham_normalize(
    input_dir: str,
    output_dir: str,
    target_size: int = 512,
    crop_thresh: int = 10,
    sigmaX_ratio: float = 0.03
) -> None:
    """
    Ben Graham normalization without any masking. Preserves 100% of retinal tissue
    using border reflection padding to eliminate boundary halo artifacts.
    Processes every image in input_dir and saves results to output_dir,
    showing a progress bar and reporting any files that failed to process.
    """
    os.makedirs(output_dir, exist_ok=True)

    valid_ext = ('.png', '.jpg', '.jpeg', '.tif', '.tiff', '.bmp')
    files = [f for f in os.listdir(input_dir) if f.lower().endswith(valid_ext)]

    failed_files = []

    pbar = tqdm(files, desc="Normalizing images", unit="img")
    for fname in pbar:
        try:
            in_path = os.path.join(input_dir, fname)
            out_path = os.path.join(output_dir, fname)

            image = cv2.imread(in_path)
            if image is None:
                raise ValueError("Failed to read image (unsupported format or corrupt file)")

            # ---- core logic (unchanged) ----
            gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
            mask = gray > crop_thresh

            if np.any(mask):
                x_idx, y_idx = np.where(mask)
                image = image[np.min(x_idx):np.max(x_idx), np.min(y_idx):np.max(y_idx)]

            resized = cv2.resize(image, (target_size, target_size), interpolation=cv2.INTER_AREA)

            pad_len = int(target_size * 0.1)
            padded = cv2.copyMakeBorder(resized, pad_len, pad_len, pad_len, pad_len, cv2.BORDER_REFLECT)

            sigmaX = max(1, int(target_size * sigmaX_ratio))
            blurred = cv2.GaussianBlur(padded, (0, 0), sigmaX)
            normalized_padded = cv2.addWeighted(padded, 4, blurred, -4, 128)

            normalized = normalized_padded[pad_len:-pad_len, pad_len:-pad_len]
            normalized = np.clip(normalized, 0, 255).astype(np.uint8)
            # ---- end core logic ----

            cv2.imwrite(out_path, normalized)

        except Exception as e:
            failed_files.append((fname, str(e)))

        pbar.set_postfix(errors=len(failed_files))

    print(f"\nDone. {len(files) - len(failed_files)}/{len(files)} images processed successfully.")

    if failed_files:
        print(f"\n⚠️ {len(failed_files)} file(s) failed to process:")
        for fname, err in failed_files:
            print(f"  - {fname}: {err}")
    else:
        print("No errors encountered.")

#APTOS

In [ ]:
INPUT_FOLDER = "/content/drive/MyDrive/Diabetic-Retinopathy/Preprocessed image/Circular Crop/1a/APTOS"
OUTPUT_FOLDER = "/content/drive/MyDrive/Diabetic-Retinopathy/Preprocessed image/Circular Crop/5a/APTOS"

In [ ]:
ben_graham_normalize(INPUT_FOLDER, OUTPUT_FOLDER)

#IDRID


In [ ]:
INPUT_FOLDER = "/content/drive/MyDrive/Diabetic-Retinopathy/Preprocessed image/Circular Crop/1a/IDRID"
OUTPUT_FOLDER = "/content/drive/MyDrive/Diabetic-Retinopathy/Preprocessed image/Circular Crop/5a/IDRID"

In [ ]:
ben_graham_normalize(INPUT_FOLDER, OUTPUT_FOLDER)

#MESSIDOR

In [ ]:
INPUT_FOLDER = "/content/drive/MyDrive/Diabetic-Retinopathy/Preprocessed image/Circular Crop/1a/MESSIDOR"
OUTPUT_FOLDER = "/content/drive/MyDrive/Diabetic-Retinopathy/Preprocessed image/Circular Crop/5a/MESSIDOR"

In [ ]:
ben_graham_normalize(INPUT_FOLDER, OUTPUT_FOLDER)